# Retail Demand Forecast — SageMaker BYOC

Entrena el modelo de predicción de demanda en SageMaker usando un contenedor propio,
despliega un endpoint de inferencia en tiempo real y genera predicciones.

In [ ]:
import io

import boto3
import pandas as pd
import sagemaker
from sagemaker import get_execution_role

sess    = sagemaker.Session()
role    = get_execution_role()
account = boto3.client("sts").get_caller_identity()["Account"]
region  = sess.boto_session.region_name
bucket  = sess.default_bucket()
prefix  = "retail-demand-forecast"

print(f"Account : {account}")
print(f"Region  : {region}")
print(f"Bucket  : {bucket}")

## 1. Subir datos de prep a S3

Los outputs de `prep.py` (`matrix.csv.gz`, `feature_cols.json`, `meta.json`) se suben al
bucket de SageMaker. El training job los va a leer desde `/opt/ml/input/data/training/`.

In [ ]:
prep_dir = "../../data/prep"

data_uri = sess.upload_data(
    path=prep_dir,
    bucket=bucket,
    key_prefix=f"{prefix}/training-data",
)

print(f"Datos subidos a: {data_uri}")

## 2. Build y push de la imagen a ECR

Correr en una terminal desde `sagemaker/container/`:

```bash
bash build_and_push.sh retail-demand-forecast
```

Una vez terminado, copiar el URI de la imagen en la celda siguiente.

In [ ]:
image_name = "retail-demand-forecast"
image_uri  = f"{account}.dkr.ecr.{region}.amazonaws.com/{image_name}:latest"

print(f"Image URI: {image_uri}")

## 3. Entrenamiento

In [ ]:
from sagemaker.estimator import Estimator

estimator = Estimator(
    image_uri=image_uri,
    role=role,
    instance_count=1,
    instance_type="ml.m5.large",
    output_path=f"s3://{bucket}/{prefix}/model-artifacts",
    sagemaker_session=sess,
    base_job_name="retail-forecast",
    hyperparameters={
        "n_estimators": 400,
        "max_depth": 8,
        "learning_rate": 0.08,
    },
)

estimator.fit({"training": data_uri}, wait=True, logs="All")

## 4. Despliegue del endpoint

In [ ]:
predictor = estimator.deploy(
    initial_instance_count=1,
    instance_type="ml.m5.large",
)

print(f"Endpoint: {predictor.endpoint_name}")

## 5. Inferencias en tiempo real

In [ ]:
test_pairs = pd.read_csv("../../data/prep/test_pairs.csv")
muestra    = test_pairs[["shop_id", "item_id"]].head(20)

print("Pares enviados:")
print(muestra)

In [ ]:
runtime = boto3.client("sagemaker-runtime", region_name=region)

response = runtime.invoke_endpoint(
    EndpointName=predictor.endpoint_name,
    ContentType="text/csv",
    Body=muestra.to_csv(index=False),
)

resultado = pd.read_csv(io.StringIO(response["Body"].read().decode("utf-8")))

print("Predicciones:")
print(resultado)

## 6. Limpieza

Eliminar el endpoint cuando ya no se necesite para evitar cargos.

In [ ]:
predictor.delete_endpoint()
print("Endpoint eliminado.")